In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

Prompt:
The source file provided contains security level data of factor elements. The goal is to use random forest to identify the most important features per stock. Once features are identified classify the stock as Value, Growth, or both (assuming most factors are relatively of equal importance). Value factors are as follows: hist_pb, hist_earn_yld, fwd_dps, fwd_dvy. Growth factors are as follows: hist_eps, hist_sps, fwd_pe, fwd_eps, fwd_eps_STg, and fwd_eps_LTg. Group the stocks based on the classification (value, growth, both) per date period and calculate a weighting scheme on each stock in the group based on the magnitude of the combined important feature elements of each stock. Set the target variable to the next monthly stock return. derived the return using the price column.


preprocess null and na values

In [2]:


# Value and growth factor definitions
value_factors = ['hist_pb', 'hist_earn_yld', 'fwd_dps', 'fwd_dvy']
growth_factors = ['hist_eps', 'hist_sps', 'fwd_pe', 'fwd_eps', 'fwd_eps_STg', 'fwd_eps_LTg']
all_factors = value_factors + growth_factors

# Load your data
df = pd.read_csv('test_source.csv')

# Ensure date column is datetime
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['stock', 'date'])

# Calculate next month's return for each stock
df['next_price'] = df.groupby('stock')['price'].shift(-1)
df['next_return'] = (df['next_price'] / df['price']) - 1

# Remove rows where next_return is NaN (usually last date per stock)
df = df.dropna(subset=['next_return'])

results = []

for date, date_df in df.groupby('date'):
    for stock, stock_df in date_df.groupby('stock'):
        # We just want the current observation to predict the next return
        # Only run RF if enough samples are available historically (say at least 12 for stability)
        hist_df = df[(df['stock'] == stock) & (df['date'] < date)]
        
        if len(hist_df) < 2:
            continue

        X = hist_df[all_factors]
        y = hist_df['next_return']
        
        if X.isnull().any().any() or y.isnull().any():
            continue  # Skip if any NA remains for features or target

        rf = RandomForestRegressor(n_estimators=100, random_state=42)
        rf.fit(X, y)
        importances = rf.feature_importances_

        # Identify important features (above mean importance)
        threshold = importances.mean()
        important_features = [f for f, imp in zip(all_factors, importances) if imp > threshold]

        # Classify
        if important_features:
            is_value = all(f in value_factors for f in important_features)
            is_growth = all(f in growth_factors for f in important_features)
            if is_value:
                classification = 'Value'
            elif is_growth:
                classification = 'Growth'
            else:
                classification = 'Both'
        else:
            classification = 'Unclassified'  # or skip

        # For the current row (date, stock), get magnitudes for important features
        curr_row = stock_df.iloc[0]
        imp_feature_sum = curr_row[important_features].abs().sum() if important_features else np.nan

        results.append({
            'date': date,
            'stock': stock,
            'classification': classification,
            'important_features': important_features,
            'imp_feature_sum': imp_feature_sum
        })
        #print(results)

# Compile results
results_df = pd.DataFrame(results).dropna(subset=['imp_feature_sum'])

# Weighting within each group per date
final_rows = []
for (date, cls), group in results_df.groupby(['date', 'classification']):
    total = group['imp_feature_sum'].sum()
    if total > 0:
        group = group.copy()
        group['weight'] = group['imp_feature_sum'] / total
        for _, row in group.iterrows():
            final_rows.append(row)

final_df = pd.DataFrame(final_rows)

# Save or display
#final_df.to_csv('stock_classification_and_weights.csv', index=False)
print(final_df.head(20))


         date           stock classification  \
0  2025-02-28     A UN Equity           Both   
1  2025-02-28  AAPL UW Equity           Both   
2  2025-02-28  ABNB UW Equity           Both   
3  2025-02-28  ACGL UW Equity           Both   
4  2025-02-28   ACN UN Equity           Both   
5  2025-02-28  ADBE UW Equity           Both   
6  2025-02-28   ADI UW Equity           Both   
7  2025-02-28   ADM UN Equity           Both   
8  2025-02-28   ADP UW Equity           Both   
9  2025-02-28  ADSK UW Equity           Both   
10 2025-02-28   AEE UN Equity           Both   
11 2025-02-28   AEP UW Equity           Both   
12 2025-02-28   AES UN Equity           Both   
13 2025-02-28   AFL UN Equity           Both   
14 2025-02-28   AIG UN Equity           Both   
15 2025-02-28   AIZ UN Equity           Both   
16 2025-02-28   AJG UN Equity           Both   
17 2025-02-28  AKAM UW Equity           Both   
18 2025-02-28  ALGN UW Equity           Both   
19 2025-02-28   ALL UN Equity           

In [5]:
final_df#.to_csv('test_results')

,date,stock,classification,important_features,imp_feature_sum,weight
0,2025-02-28,A UN Equity,Both,"[hist_pb, hist_earn_yld, fwd_dvy, fwd_pe, fwd_...",32.704832,0.001113
1,2025-02-28,AAPL UW Equity,Both,"[hist_pb, hist_earn_yld, fwd_dvy, hist_sps, fw...",115.414472,0.003928
2,2025-02-28,ABNB UW Equity,Both,"[hist_pb, hist_earn_yld, fwd_pe, fwd_eps, fwd_...",49.372340,0.001680
3,2025-02-28,ACGL UW Equity,Both,"[hist_pb, hist_earn_yld, fwd_dvy, fwd_pe, fwd_...",25.144718,0.000856
4,2025-02-28,ACN UN Equity,Both,"[hist_pb, hist_earn_yld, fwd_dvy, fwd_pe, fwd_...",38.122802,0.001297
...,...,...,...,...,...,...
1170,2025-04-30,NWSA UW Equity,Value,"[hist_pb, hist_earn_yld, fwd_dvy]",4.960611,0.044111
1173,2025-04-30,ODFL UW Equity,Value,"[fwd_dps, fwd_dvy]",1.160378,0.010318
1178,2025-04-30,OXY UN Equity,Value,"[hist_pb, hist_earn_yld, fwd_dvy]",10.086058,0.089687
1198,2025-04-30,PODD UW Equity,Value,"[hist_pb, hist_earn_yld]",15.899894,0.141385
